In [1]:
import sys
import numpy as np
import pyproj
from pyproj import Transformer

# Euler pole calculation

#### Author: Yuan-Kai Liu (updated 2022-08-04)
A brief docstring about the math/geometry is written in a [LaTex document](https://www.overleaf.com/project/62dbfabf8aef8a343c7210e8)

#### Useful reference:
+ Pichon, X. L., Francheteau, J. & Bonnin, J. Plate Tectonics; Developments in Geotectonics 6; Hardcover – January 1, 1973. Page 28-29

+ Cox, A., and Hart, R.B. (1986) Plate tectonics: How it works. Blackwell Scientific Publications, Palo Alto. DOI: 10.4236/ojapps.2015.54016. Page 145-156.

+ [Transformations between ECEF and ENU coordinates](https://gssc.esa.int/navipedia/index.php/Transformations_between_ECEF_and_ENU_coordinates)



In [6]:
# Given a pole and angular velocity, find the Cartesian rotation vector

EARTH_RADIUS = 6371.009           #  km (Mean radius from IUGG)
MAS2RAD      = np.pi/3600000/180  #  1 mas (milliarcsecond) = yy radian
MASY2DMY     = 1e6 / 3600000      #  1 mas per year         = xx degree per million year


def sph2cart(lat, lon, r=1):
    """Convert spherical coordinates to cartesian. Default raduis is 1 (unit length).
    INPUT
        lat     latitude   [degree]
        lon     longitude  [degree]
        r       radius     [any units of distance]
    OUTPUT
        u = ndarray([u1, u2, u3])      cartesian vector    [same unit as r]
    """
    lat, lon = np.deg2rad(lat), np.deg2rad(lon)
    u1 = r * np.cos(lat) * np.cos(lon)       # in x-axis
    u2 = r * np.cos(lat) * np.sin(lon)       # in y-axis
    u3 = r * np.sin(lat)                     # in z-axis
    u  = np.array([u1, u2, u3]).T
    return u


def cart2sph(u):
    """Convert cartesian coordinates to spherical.
    INPUT
        u = ndarray([u1, u2, u3])     cartesian vector     [any units of distance]
    OUTPUT
        lat     latitude   [degree]
        lon     longitude  [degree]
        r       radius     [same unit as u]
    """    
    u1, u2, u3 = u
    r   = np.sqrt(u1**2+u2**2+u3**2)    
    lat = np.arcsin(u3/r)
    lon = np.arctan2(u2, u1)
    lat = np.rad2deg(lat)
    lon = np.rad2deg(lon)
    return lat, lon, r


def epole2Omega(lat, lon, omega=1):
    """Convert Euler pole (in spherical) to cartesian Omega (angular velocity vector).
       Default lat lon input in degree.
       Output Omega whatever input units are for the omega (scalar angular velocity).
    INPUT
        lat      latitude                [degree]
        lon      longitude               [degree]
        omega    scalar angular velocity [any units, e.g., milliarcsec (mas) per year]
    OUTPUT
        Omega    angular velocity vector [same unit as omega]
    """
    u = sph2cart(lat, lon, r=1)
    Omega = omega * u
    return Omega


def T_cart2enu(lats, lons):
    """Rotation matrix to convert cartesian coordinates (global ECEF) at given (lats,lons) to ENU components (local cartesian).
       Input lats lons are in degree. If input N points of (lats,lons), T.shape = (N, 3, 3)
    INPUT
        lats    1d-array of latitude      [degree]
        lons    1d-array of longitude     [degree]
    OUTPUT
        T       rotation matrix (N, 3, 3) [-]
    REFERENCE 
        + https://gssc.esa.int/navipedia/index.php/Transformations_between_ECEF_and_ENU_coordinates
        + Cox, A., and Hart, R.B. (1986) Plate tectonics: How it works. Blackwell Scientific Publications, Palo Alto. 
          (DOI: 10.4236/ojapps.2015.54016. Page 145-156)
    """
    lats = np.deg2rad(lats)
    lons = np.deg2rad(lons)
    if not isinstance(lats,(list,np.ndarray)):
        lats = [lats]
        lons = [lons]
    if not len(lats) == len(lons):
        print('Dimension is not the same')
        sys.exit(1)
    else:
        npts = len(lats)
        Te = np.vstack((-np.sin(lons), np.cos(lons), np.zeros(npts))).T
        Tn = np.vstack((-np.sin(lats)*np.cos(lons), -np.sin(lats)*np.sin(lons), np.cos(lats))).T
        Tu = np.vstack((np.cos(lats)*np.cos(lons), np.cos(lats)*np.sin(lons), np.sin(lats))).T
        T  = np.dstack((Te, Tn, Tu))
        T  = np.transpose(T, (0, 2, 1))
    return T

    
def T_enu2cart(lats, lons):
    """Rotation matrix to convert ENU components (local cartesian) at given (lats,lons) to cartesian coordinates (global ECEF).
       Input lats lons are in degree. If input N points of (lats,lons), T.shape = (N, 3, 3)
    INPUT
        lats    1d-array of latitude      [degree]
        lons    1d-array of longitude     [degree]
    OUTPUT
        T       rotation matrix (N, 3, 3) [-]
    REFERENCE
        + https://gssc.esa.int/navipedia/index.php/Transformations_between_ECEF_and_ENU_coordinates
        + Cox, A., and Hart, R.B. (1986) Plate tectonics: How it works. Blackwell Scientific Publications, Palo Alto. 
          (DOI: 10.4236/ojapps.2015.54016. Page 145-156)
    """
    lats = np.deg2rad(lats)
    lons = np.deg2rad(lons)
    if not isinstance(lats,(list,np.ndarray)):
        lats = [lats]
        lons = [lons]
    if not len(lats) == len(lons):
        print('Dimension is not the same')
        sys.exit(1)
    else:
        npts = len(lats)
        print('{} points'.format(npts))
        Te = np.vstack((-np.sin(lons), -np.cos(lons)*np.sin(lats), np.cos(lons)*np.cos(lats))).T
        Tn = np.vstack((np.cos(lons), -np.sin(lons)*np.sin(lats), np.sin(lons)*np.cos(lats))).T
        Tu = np.vstack((np.zeros(npts), np.cos(lats), np.sin(lats))).T
        T  = np.dstack((Te, Tn, Tu))
        T  = np.transpose(T, (0, 2, 1))
    return T


def azimuth(east, north):
    """Returns azimuth in degrees counterclockwise from North given north and
    east components"""
    azi = np.rad2deg(np.arctan2(north, east))
    azi = 90 - azi
    if azi <= 0: 
        azi +=360
    return azi


def geo2ecef_pyproj(lat, lon, alt):
    transformer = pyproj.Transformer.from_crs(
        {"proj":'latlong', "ellps":'WGS84', "datum":'WGS84'},
        {"proj":'geocent', "ellps":'WGS84', "datum":'WGS84'},
        )
    x ,y, z = transformer.transform(lon, lat, alt, radians=False)
    return x, y, z


def epole2Venu(lats, lons, alts=0.0, plat=None, plon=None, omega=None, Omega=None, ellps=True):
    """Given Euler pole, compute V_enu for given pixel(s) of interest.
       Only supports uniform altitude now.
       The unit of angular velocity must be "mas per year".
    INPUT
        lats      points of interest (latitude)       [degree]
        lons      points of interest (longitude)      [degree]
        alts      points of interest (altitude)       [degree]

        Euler pole, option (1): spherical description
            plat      Euler pole latitude             [degree]
            plon      Euler pole longitude            [degree]
            omega     scalar angular velocity         [mas per year]
        
        Euler pole, option (2): cartesian description            
            Omega     angular velocity vector         [mas per year]

        (option 2 overwrites option 1; need to specify either one of them)
    OUTPUT
        V_enu     east, north, up linear velocity     [mm/yr]
    """
    ## Euler pole handling
    if Omega is not None:
        print('\nUser keyin Euler pole in Cartesian coord')
        plat, plon, omega = cart2sph(Omega)
    elif all(k is not None for k in [plat, plon, omega]):
        print('\nUser keyin Euler pole in Spherical coord')
        Omega = epole2Omega(plat, plon, omega)
    else:
        print('Wrong user input')
        sys.exit(1)
    print('--------------------------------------------------------------------------------------')
    print('Euler pole (spherical)             : {:8.4f}N {:8.4f}E {:8.4} mas/yr'.format(plat, plon, omega))
    print('Angular velocity vector (cartesian): {:8.4f} {:8.4f} {:8.4f} mas/yr'.format(*Omega))
    print('--------------------------------------------------------------------------------------')
    
    ## report how many points of interest
    if isinstance(lats, (list, tuple, np.ndarray)):
        npts = len(lats)
    elif isinstance(lats, (int, float)):
        npts = 1
    print('number of points to compute: {}'.format(npts))
    
    ## Local coordinates handling for location(s) of interest
    if not ellps:
        # a perfect sphere
        print('Assume a perfect spherical Earth, radius: {} km'.format(EARTH_RADIUS))
        locs_xyz = sph2cart(lats, lons, EARTH_RADIUS)                    # unit is km
    else:
        # WGS84 ellips; only supports uniform altitude now, but can change later
        if npts == 1:
            alts = float(alts)
        else:
            alts = alts * np.ones_like(lats)
        print('Assume WGS84 ellipse from pyproj')
        locs_xyz = 1e-3 * np.array(geo2ecef_pyproj(lats, lons, alts)).T  # set unit as km

    ## Compute the cartesian linear velocity (i.e., ECEF)
    V_ecef = np.cross(Omega*MAS2RAD, locs_xyz)                           # watch out unit here!

    ## Convert to local ENU linear velocity
    T = T_cart2enu(lats, lons)    # the rotation matrix (can save it to avoid computing again)
    V_tmp = np.matmul(T.reshape([-1,3]) , V_ecef.T).reshape([3,npts,npts], order='F')
    V_enu = 1e6 * np.diagonal(V_tmp, axis1=1, axis2=2).T                 # set unit as mm/year
    del V_tmp
    return V_enu, T

## Test the input angular velocity vector (Omega) and convert to Euler pole (lat, lon, omega)

In [7]:
## Test a point on Eurasian plate

# get from Altamimi (2017) Table 1. EURA plate
EURA = np.array([-0.085, -0.531, 0.770]) # unit: mas per year
print('My input Omega (Eurasian plate): ', EURA)

# test converting it to spherical Euler pole
plat, plon, omega = cart2sph(EURA)
# print the rotation rate as deg/Ma (compare with Altamimi (2017) Table 1)
print('Convert my input to Euler pole and rotation rate:')
print('    Euler pole lat  : ', plat)
print('    Euler pole lon  : ', plon)
print('    Angular velocity: ', omega*MASY2DMY)


# convert back to angular velocity vector (check if it is consistent as earlier input)
test_EURA = epole2Omega(plat, plon, omega)
print('Convert the Euler pole back to Omega: ', test_EURA)  # it is consistent with my input


My input Omega (Eurasian plate):  [-0.085 -0.531  0.77 ]
Convert my input to Euler pole and rotation rate:
    Euler pole lat  :  55.06994336719982
    Euler pole lon  :  -99.09448520428982
    Angular velocity:  0.2608873154410246
Convert the Euler pole back to Omega:  [-0.085 -0.531  0.77 ]


## Compute linear velocity from a given Euler pole (test consistency with UNAVCO calculator)

The Eurasian plate motion in ITRF2014 provided by Altamimi 2017 (unit: $mas \cdot yr^{-1}$):

| $\omega_x$ | $\omega_y$ | $\omega_z$ |
|------------|------------|------------|
| -0.085     |  -0.531    | 0.770      |

Using the UNAVCO [Plate motion Calculator](https://www.unavco.org/software/geodetic-utilities/plate-motion-calculator/plate-motion-calculator.html), we can compute the east, north, up velocity from the locations of interest. Here we check the following locations.

+ Latitue $50.0^{\circ}$ N, Longitude $10.0^{\circ}$ E

    N velocity [mm/yr] = 15.68

    E velocity [mm/yr] = 19.48
    
    
+ Latitue $20.0^{\circ}$ N, Longitude $45.0^{\circ}$ E

    N velocity [mm/yr] = 9.75

    E velocity [mm/yr] = 26.96    

In [8]:
## Result: consistent with unavco calculator 
## Test two locations of interest lat,lon = (50., 10.), (20., 45.) on EURA plate

ground_truth = [[19.48, 15.68], [26.96, 9.75]]  # from unavco calculator

# locations coord
lats = (50., 20.)
lons = (10., 45.)

def print_venu(venu, truth):
    npts = len(venu)
    for i in range(npts):
        rms = np.sqrt(((venu[i][0]-truth[i][0])**2 + (venu[i][1]-truth[i][1])**2)/2)
        print(' Location {}: ({:7.3f} mm/yr E, {:7.3f} mm/yr N, {:7.3f} mm/yr U )  --> RMS= {:7.4f} mm/yr'.format(i+1, *venu[i], rms))
    print('\n')
    

## Test with or without using the WGS ellipse to describe the locations of interest

#   try using the spherical description of Euler pole
venu = epole2Venu(lats=lats, lons=lons, plat=plat, plon=plon, omega=omega)[0]
print_venu(venu, ground_truth)
venu = epole2Venu(lats=lats, lons=lons, plat=plat, plon=plon, omega=omega, ellps=False)[0]
print_venu(venu, ground_truth)


#   try using the cartesian description of Euler pole
venu = epole2Venu(lats=lats, lons=lons, Omega=EURA)[0]
print_venu(venu, ground_truth)
venu = epole2Venu(lats=lats, lons=lons, Omega=EURA, ellps=False)[0]
print_venu(venu, ground_truth)


User keyin Euler pole in Spherical coord
--------------------------------------------------------------------------------------
Euler pole (spherical)             :  55.0699N -99.0945E   0.9392 mas/yr
Angular velocity vector (cartesian):  -0.0850  -0.5310   0.7700 mas/yr
--------------------------------------------------------------------------------------
number of points to compute: 2
Assume WGS84 ellipse from pyproj
 Location 1: ( 19.482 mm/yr E,  15.683 mm/yr N,   0.052 mm/yr U )  --> RMS=  0.0026 mm/yr
 Location 2: ( 26.960 mm/yr E,   9.748 mm/yr N,   0.021 mm/yr U )  --> RMS=  0.0014 mm/yr



User keyin Euler pole in Spherical coord
--------------------------------------------------------------------------------------
Euler pole (spherical)             :  55.0699N -99.0945E   0.9392 mas/yr
Angular velocity vector (cartesian):  -0.0850  -0.5310   0.7700 mas/yr
--------------------------------------------------------------------------------------
number of points to compute: 2
Ass

### Summary

In general, the RMS difference between our estimate and the ground truth from UNAVCO is at the order smaller than 0.03 mm/yr (<0.1$%$ of actual velocity magnitude). Thus, the code here is consistent and should be fine. Not sure if this is just the floating points/precision/rounding reason.

### About sphere vs ellipse results

The difference between the equatorial and polar radii of the WGS84 ellipsoid is only around 21 km, so I think that difference will be insignificant for general purposes (tectonic problems). [There are some more discussion of that online](https://link.springer.com/article/10.1007/s10291-013-0354-4#Sec8).

Here we directly estimate the error by looking at the difference in velocity we get when we take an Euler pole and use that to get velocities on a spherical surface vs ellipsoidal surface. The differences turn out to be small at given locations. The UNAVCO calculator is using the WGS84 ellipsoid. Thus, by using a WGS84 here gives even smaller RMS difference.

### Where is the ellipse been used

Once you have an Euler pole, you can use it to calculate velocities anywhere you like in the space. The ellipsoid is only used in converting the locations of interest (lat, lon) on Earth's reference surface to their Cartesian coordinates (x, y, z) in a ECEF frame. So that we can estimate the linear velocity at that point on Earth.

Note that, once we get that linear velocity $V_{xyz}$, whether we use the ellipsoid or not does not change the rotation matrix from global ECEF linear velocity ($V_{xyz}$) to the local ENU velocity $V_{ENU}$. The matrix rotation is the same. The difference is either you are solving a local ENU plane which is tagential to the a sphere or to an ellipsoid at that specific point (lat, lon). You can read the [transformation between ECEF and ENU coordinates](https://gssc.esa.int/navipedia/index.php/Transformations_between_ECEF_and_ENU_coordinates).

The Euler theorem about the plate motion model relies on the key assumption of a perfect sphere (it CANNOT be an ellipsoid; the trajectory of a plate moving on ellipsoidal Earth  is a parabola, very complicated). Thus, the key assumption comes when you (and studies about plate motion models) take a bunch of measured ground velocities and use that to calculate the Euler pole - this relies on the spherical earth assumption. [And it should be insignificant](https://link.springer.com/article/10.1007/s10291-013-0354-4#Sec8).

### Vertical motions

We should know that the vertical velocities at a point on Earth's surface do not contribute to the estimation of the Euler pole parameters. As we can see from [equation (5) here](https://www.overleaf.com/project/62dbfabf8aef8a343c7210e8) or [equation (10) here](https://link.springer.com/article/10.1007/s10291-013-0354-4) that this is the same for both forward or inverse Euler pole problems. Thus, given an Euler pole, if we compute velocities on a ellipsoidal Earth, we will end up having some vertival motions. Ignore that. See more discussions [here](https://link.springer.com/article/10.1007/s10291-013-0354-4#Sec11).

## Examples of computing relative velocity between two plates

In [9]:
# Grab the Euler poles from Altamimi's paper (these are the cartesian rotation components)
arab  = np.array([1.154, -0.136, 1.444])
nubi  = np.array([0.099, -0.614, 0.733])
sinai = epole2Omega(54.7, 347.8, 0.417/MASY2DMY)

# The rotation Euler poles of these two plates
print('\nEuler pole: ARABIA')
print(cart2sph(arab))
print('\nEuler pole: NUBIA')
print(cart2sph(nubi))
print('\nEuler pole: Sinai')
print(cart2sph(sinai))


# Relative rotation between two plates
arab_nubi = arab-nubi
print('\nRelative rotation: ARAB-NUBI')
print(cart2sph(arab_nubi))

# convert from mas/yr to deg/Ma
print('\nRelative rotation rate: ARAB-NUBI')
print(cart2sph(arab_nubi)[-1], '[mas/yr]')    
print(cart2sph(arab_nubi)[-1]*MASY2DMY, '[deg/Ma]')

arab_sinai = arab-sinai
print('\nRelative rotation: ARAB-SINAI')
print(cart2sph(arab_sinai))

# convert from mas/yr to deg/Ma
print('\nRelative rotation rate: ARAB-SINAI')
print(cart2sph(arab_sinai)[-1], '[mas/yr]')    
print(cart2sph(arab_sinai)[-1]*MASY2DMY, '[deg/Ma]')


Euler pole: ARABIA
(51.176380187661, -6.721359337418204, 1.8534691796736193)

Euler pole: NUBIA
(49.68632402805456, -80.84058683616999, 0.9612939196728543)

Euler pole: Sinai
(54.699999999999996, -12.200000000000017, 1.5011999999999999)

Relative rotation: ARAB-NUBI
(31.544303466638066, 24.374356511559867, 1.3590548186147606)

Relative rotation rate: ARAB-NUBI
1.3590548186147606 [mas/yr]
0.3775152273929891 [deg/Ma]

Relative rotation: ARAB-SINAI
(35.23846747435657, 8.787470805678229, 0.3792400858821064)

Relative rotation rate: ARAB-SINAI
0.3792400858821064 [mas/yr]
0.10534446830058511 [deg/Ma]


## Testing speed for multiple points

Given a arbitrary Euler pole and estimate the Venu at some random locations

In [5]:
%%time

lats = np.arange(0,90)
lons = np.arange(0,18)
lats, lons = np.meshgrid(lons, lats)
lats = lats.flatten()
lons = lons.flatten()

_, _ = epole2Venu(lats, lons, plat=60, plon=30, omega=15)


User keyin Euler pole in Spherical coord
------------------------------------------------------
Euler pole (spherical):  30 60 15
Angular velocity vector (cartesian):  6.495190528383292 11.25 7.499999999999999
------------------------------------------------------
number of points to compute: 1620
CPU times: user 72.7 ms, sys: 79.4 ms, total: 152 ms
Wall time: 68.2 ms


In [6]:
%%time

lats = np.arange(0,90)
lons = np.arange(0,18)
lats, lons = np.meshgrid(lons, lats)
lats = lats.flatten()
lons = lons.flatten()

_, _ = epole2Venu(lats, lons, plat=60, plon=30, omega=15, ellps=False)


User keyin Euler pole in Spherical coord
------------------------------------------------------
Euler pole (spherical):  30 60 15
Angular velocity vector (cartesian):  6.495190528383292 11.25 7.499999999999999
------------------------------------------------------
number of points to compute: 1620
CPU times: user 53.5 ms, sys: 60.4 ms, total: 114 ms
Wall time: 31.9 ms


In [8]:
%%time

lats = np.arange(0,90)
lons = np.arange(0,180)
lats, lons = np.meshgrid(lons, lats)
lats = lats.flatten()
lons = lons.flatten()

_, _ = epole2Venu(lats, lons, plat=60, plon=30, omega=15)


User keyin Euler pole in Spherical coord
------------------------------------------------------
Euler pole (spherical):  30 60 15
Angular velocity vector (cartesian):  6.495190528383292 11.25 7.499999999999999
------------------------------------------------------
number of points to compute: 16200
CPU times: user 2.78 s, sys: 5.43 s, total: 8.21 s
Wall time: 2.09 s


In [9]:
%%time

lats = np.arange(0,90)
lons = np.arange(0,180)
lats, lons = np.meshgrid(lons, lats)
lats = lats.flatten()
lons = lons.flatten()

_, _ = epole2Venu(lats, lons, plat=60, plon=30, omega=15, ellps=False)


User keyin Euler pole in Spherical coord
------------------------------------------------------
Euler pole (spherical):  30 60 15
Angular velocity vector (cartesian):  6.495190528383292 11.25 7.499999999999999
------------------------------------------------------
number of points to compute: 16200
CPU times: user 2.66 s, sys: 5.47 s, total: 8.13 s
Wall time: 2.05 s


In [10]:
%%time

lats = np.arange(0,90)
lons = np.arange(-180,180)
lats, lons = np.meshgrid(lons, lats)
lats = lats.flatten()
lons = lons.flatten()

_, _ = epole2Venu(lats, lons, plat=60, plon=30, omega=15)


User keyin Euler pole in Spherical coord
------------------------------------------------------
Euler pole (spherical):  30 60 15
Angular velocity vector (cartesian):  6.495190528383292 11.25 7.499999999999999
------------------------------------------------------
number of points to compute: 32400
CPU times: user 13 s, sys: 2min 8s, total: 2min 21s
Wall time: 42 s


In [11]:
%%time

lats = np.arange(0,90)
lons = np.arange(-180,180)
lats, lons = np.meshgrid(lons, lats)
lats = lats.flatten()
lons = lons.flatten()

_, _ = epole2Venu(lats, lons, plat=60, plon=30, omega=15, ellps=False)


User keyin Euler pole in Spherical coord
------------------------------------------------------
Euler pole (spherical):  30 60 15
Angular velocity vector (cartesian):  6.495190528383292 11.25 7.499999999999999
------------------------------------------------------
number of points to compute: 32400
CPU times: user 11.3 s, sys: 56.3 s, total: 1min 7s
Wall time: 19.8 s


In [ ]:
%%time

lats = np.arange(-90,90)
lons = np.arange(-180,180)
lats, lons = np.meshgrid(lons, lats)
lats = lats.flatten()
lons = lons.flatten()

_, _ = epole2Venu(lats, lons, plat=60, plon=30, omega=15)

# Todo: 

#### 1. Given a set of ENU velocity points, invert for the Euler pole (lat, lon, omega)

This is done a lot in GNSS/GPS community. Try to write our own version and apply it in InSAR velocity field.

We can probably check this: https://github.com/USFgeodesy/usfEP


#### 2. Add a plotting function to plot the plate polygon, local ENU as quivers, also show the location of the pole if visible.

In [13]:
# Compute the distance between pole and location

from pyproj import Geod
g = Geod(ellps='WGS84')

# Grab the Euler poles from Altamimi's paper (these are the cartesian rotation components)
arab  = np.array([1.154, -0.136, 1.444])
eura  = np.array([-0.085, -0.531, 0.770])
aust  = np.array([1.510, 1.182, 1.215])

# The rotation Euler poles of these two plates
print('\nEuler pole: ARABIA')
print(cart2sph(arab))
print('\nEuler pole: EURASIA')
print(cart2sph(eura))
print('\nEuler pole: AUSTRALIA')
print(cart2sph(aust))

aqab_lat, aqab_lon = 30, 36
makr_lat, makr_lon = 29, 61
aust_lat, aust_lon = 25, 124


Euler pole: ARABIA
(51.176380187661, -6.721359337418204, 1.8534691796736193)

Euler pole: EURASIA
(55.06994336719982, -99.09448520428982, 0.9391943355876886)

Euler pole: AUSTRALIA
(32.35841061673232, 38.05318070766016, 2.2701209218894047)


In [14]:
print(g.inv(-6.721359337418204, 51.176380187661,   aqab_lon, aqab_lat))
print(g.inv(-99.09448520428982, 55.069943367199,   makr_lon, makr_lat))
print(g.inv(38.05318070766016,  32.35841061673232, aust_lon, aust_lat))

(107.13127810951103, -43.83801597186988, 4226830.836015361)
(17.34423939434981, -11.270881528996597, 10497945.43518503)
(70.30981755069199, -61.38539861113709, 8212366.051353086)
